# Cell counts per FTU per dataset: Anatomogram and HRApop

This notebook compares the distribution of cells associated with each FTU across datasets in the Anatomogram and HRApop resources.

For Anatomogram, the author annotation is used so that each cell is counted once. For HRApop, each dataset is represented once using the annotation-method preference used by the FTU-to-dataset mapping. FTU membership is defined by exact CL-ID membership in `cell-types-in-ftus.json`.

## 1. Imports and configuration

In [33]:
# -----------------------------------------------------------------------------
# 1. Imports and configuration
# -----------------------------------------------------------------------------

from pathlib import Path
import json
import gzip
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# -----------------------------------------------------------------------------
# Input files
# -----------------------------------------------------------------------------

CT_FILE = Path("cell-types-in-ftus.json")
FTU_DATASET_FILE = Path("ftu_to_datasets.json")
ANATOMOGRAM_FILE = Path(
    "anatomogram-sc-transcriptomics-cell-instances.csv.gz"
)
HRAPOP_FILE = Path(
    "sc-transcriptomics-cell-summaries.jsonl.gz"
)
DATASET_METADATA_FILE = Path(
    "sc-transcriptomics-dataset-metadata.csv"
)


# -----------------------------------------------------------------------------
# Output directory
# -----------------------------------------------------------------------------

OUTPUT_DIR = Path(
    "ftu_cell_count_both_sources"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------------------------------
# Robust project-local input discovery
#
# The notebook may be launched from anatomogram-comparison, the repository
# root, or another notebook directory. We therefore search the current
# directory, its parents, the Downloads folder, common data-preprocessor
# locations, and finally the repository tree for an exact filename.
# -----------------------------------------------------------------------------

def resolve_input(path, filename):
    """Resolve an input file from common local/project locations."""

    cwd = Path.cwd().resolve()
    home = Path.home().resolve()

    candidates = [
        # Explicit path supplied by the notebook configuration.
        path,
        path.expanduser(),

        # Current and parent directories.
        cwd / filename,
        cwd.parent / filename,

        # Common project data directories.
        cwd / "data-preprocessor" / "input" / filename,
        cwd / "data-preprocessor" / "output" / filename,
        cwd.parent / "data-preprocessor" / "input" / filename,
        cwd.parent / "data-preprocessor" / "output" / filename,
        cwd / "data_preprocessor" / "input" / filename,
        cwd / "data_preprocessor" / "output" / filename,
        cwd.parent / "data_preprocessor" / "input" / filename,
        cwd.parent / "data_preprocessor" / "output" / filename,

        # Other project locations used by the Anatomogram workflow.
        cwd / "anatomogram-data" / filename,
        cwd.parent / "anatomogram-data" / filename,
        cwd / "validation" / filename,
        cwd.parent / "validation" / filename,

        # Downloads.
        home / "Downloads" / filename,
        home / "downloads" / filename,

        # ChatGPT/container runtime, when applicable.
        Path("/mnt/data") / filename,
    ]

    seen = set()

    # Also inspect the current directory and all parents for the common
    # project folder names. This handles notebooks launched from nested dirs.
    for root in [cwd, *cwd.parents]:
        candidates.extend([
            root / filename,
            root / "data-preprocessor" / "input" / filename,
            root / "data-preprocessor" / "output" / filename,
            root / "data_preprocessor" / "input" / filename,
            root / "data_preprocessor" / "output" / filename,
            root / "anatomogram-data" / filename,
            root / "validation" / filename,
        ])

    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except OSError:
            continue

        if candidate in seen:
            continue

        seen.add(candidate)

        if candidate.is_file():
            return candidate

    # Final fallback: recursively search the most likely repository root.
    # This is only used when the common locations above fail.
    repo_candidates = [cwd, cwd.parent]

    for repo_root in repo_candidates:
        try:
            for candidate in repo_root.rglob(filename):
                if candidate.is_file():
                    return candidate.resolve()
        except OSError:
            continue

    raise FileNotFoundError(
        f"Could not find {filename}. Checked common project locations and "
        f"the repository tree below {cwd}.\n\n"
        f"Set the corresponding *_FILE variable near the top of this cell "
        f"to the exact path if the file is outside the project tree."
    )


CT_FILE = resolve_input(
    CT_FILE,
    CT_FILE.name,
)

FTU_DATASET_FILE = resolve_input(
    FTU_DATASET_FILE,
    FTU_DATASET_FILE.name,
)

ANATOMOGRAM_FILE = resolve_input(
    ANATOMOGRAM_FILE,
    ANATOMOGRAM_FILE.name,
)

HRAPOP_FILE = resolve_input(
    HRAPOP_FILE,
    HRAPOP_FILE.name,
)


try:
    DATASET_METADATA_FILE = resolve_input(
        DATASET_METADATA_FILE,
        DATASET_METADATA_FILE.name,
    )
except FileNotFoundError:
    DATASET_METADATA_FILE = None


# -----------------------------------------------------------------------------
# HRApop annotation-method preference used by the FTU-to-dataset mapping.
# -----------------------------------------------------------------------------

ANNOTATION_METHOD_PREFERENCE = [
    "azimuth",
    "celltypist",
    "popv",
    "frmatch",
    "pan-human-azimuth",
]


# -----------------------------------------------------------------------------
# Publication-friendly colors.
#
# Color represents ORGAN. Data source is represented by marker shape in the
# interactive visualization.
# -----------------------------------------------------------------------------

ORGAN_COLORS = {
    "kidney": "#0072B2",
    "liver": "#D55E00",
    "lung": "#009E73",
    "pancreas": "#E69F00",
    "prostate gland": "#CC79A7",
    "skin": "#E69F00",
    "spleen": "#56B4E9",
}


print("Anatomogram:", ANATOMOGRAM_FILE)
print("HRApop:", HRAPOP_FILE)
print("FTU -> CT:", CT_FILE)
print("FTU -> dataset:", FTU_DATASET_FILE)
print("Output:", OUTPUT_DIR.resolve())


Anatomogram: /Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/anatomogram-data/anatomogram-sc-transcriptomics-cell-instances.csv.gz
HRApop: /Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/data-preprocessor/input/sc-transcriptomics-cell-summaries.jsonl.gz
FTU -> CT: /Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/data-preprocessor/output/cell-types-in-ftus.json
FTU -> dataset: /Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/data-preprocessor/output/ftu_to_datasets.json
Output: /Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/validation/ftu_cell_count_both_sources


## 2. Helper functions

In [34]:
# -----------------------------------------------------------------------------
# 2. Helper functions
# -----------------------------------------------------------------------------

def normalize_cl_id(value):
    """Normalize CL identifiers to the CL:NNNNNN form."""

    if value is None or pd.isna(value):
        return ""

    text = str(value).strip()

    match = re.search(
        r"CL[:_](\d+)$",
        text,
        flags=re.IGNORECASE,
    )

    if match:
        return "CL:" + match.group(1)

    return text


def dataset_display_name(dataset_id):
    """Return a compact dataset name for hover text and tables."""

    text = str(dataset_id)

    if "#" in text:
        return text.split("#", 1)[1]

    return text.rstrip("/").rsplit("/", 1)[-1]


def normalize_organ(value):
    if value is None or pd.isna(value):
        return ""

    return str(value).strip().lower()

## 3. Load FTU → CT definitions

In [35]:
# -----------------------------------------------------------------------------
# 3. Load FTU -> CT definitions
# -----------------------------------------------------------------------------

with CT_FILE.open(encoding="utf-8") as handle:
    cell_types_in_ftus = json.load(handle)


ftu_by_purl = {}

for ftu_label, record in cell_types_in_ftus.items():

    ftu_purl = record.get("ftu_purl", "")

    if not ftu_purl:
        continue

    ct_by_id = {}

    for item in record.get("cts_in_2d_ftu", []):

        ct_id = normalize_cl_id(item.get("ct_iri"))
        ct_label = str(item.get("ct_label", "")).strip()

        if ct_id:
            ct_by_id[ct_id] = ct_label

    ftu_by_purl[ftu_purl] = {
        "ftu_label": ftu_label,
        "organ": str(record.get("organ_label", "")).strip(),
        "organ_id": str(record.get("organ_id_short", "")).strip(),
        "ct_by_id": ct_by_id,
    }


print(f"FTU definitions loaded: {len(ftu_by_purl):,}")

FTU definitions loaded: 27


## 4. Load FTU → dataset mappings


In [36]:
# -----------------------------------------------------------------------------
# 4. Load FTU -> dataset mappings
# -----------------------------------------------------------------------------

with FTU_DATASET_FILE.open(encoding="utf-8") as handle:
    ftu_to_datasets = json.load(handle)


# These are the FTU PURLs for which both the FTU definition and HRApop
# dataset mapping are available.
matched_ftu_purls = [
    purl
    for purl in ftu_to_datasets
    if purl in ftu_by_purl
]


# We will identify which of these FTUs are also represented in Anatomogram
# using the organ ID of the FTU and the author-annotated Anatomogram cells.
print(
    f"FTU -> dataset mappings: {len(ftu_to_datasets):,}"
)
print(
    f"Mappings with a matching FTU definition: {len(matched_ftu_purls):,}"
)

FTU -> dataset mappings: 11
Mappings with a matching FTU definition: 10


## 5. Load Anatomogram author annotations

In [37]:
# -----------------------------------------------------------------------------
# 5. Load Anatomogram author annotations
#
# We use tool == "author" so each biological cell is represented once rather
# than once for every CTann method.
# -----------------------------------------------------------------------------

anatomogram = pd.read_csv(
    ANATOMOGRAM_FILE,
    compression="gzip",
    dtype=str,
    usecols=[
        "dataset",
        "organ",
        "tool",
        "cell",
        "cell_id",
        "cell_label",
    ],
)


author = anatomogram.loc[
    anatomogram["tool"].eq("author")
].copy()


# Defensive duplicate check.
duplicate_author_cells = (
    author.duplicated(
        subset=["dataset", "cell"],
        keep=False,
    ).sum()
)


print(f"Anatomogram rows: {len(anatomogram):,}")
print(f"Author-annotated cells: {len(author):,}")
print(
    "Duplicate dataset/cell author rows:",
    duplicate_author_cells,
)
print(
    "Anatomogram datasets:",
    author["dataset"].nunique(),
)

Anatomogram rows: 147,144
Author-annotated cells: 40,285
Duplicate dataset/cell author rows: 0
Anatomogram datasets: 24


## 6. Read only the HRApop datasets used by the FTU mapping

In [38]:
# -----------------------------------------------------------------------------
# 6. Read only the HRApop dataset records used by ftu_to_datasets.json.
# This avoids processing thousands of unrelated HRApop dataset summaries.
# -----------------------------------------------------------------------------

needed_hrapop_dataset_ids = {
    dataset_id
    for purl in matched_ftu_purls
    for dataset_id in set(ftu_to_datasets[purl])
}


hrapop = {}
annotation_methods = {}


with gzip.open(
    HRAPOP_FILE,
    "rt",
    encoding="utf-8",
) as handle:

    for line in handle:

        record = json.loads(line)

        dataset_id = str(
            record.get("cell_source", "")
        ).strip()

        method = str(
            record.get("annotation_method", "")
        ).strip().lower()

        if dataset_id not in needed_hrapop_dataset_ids:
            continue

        if not method:
            continue

        summary_by_ct = {}

        for summary_row in record.get("summary", []):

            ct_id = normalize_cl_id(
                summary_row.get("cell_id")
            )

            if not ct_id:
                continue

            try:
                count = float(
                    summary_row.get("count", 0)
                )
            except (TypeError, ValueError):
                continue

            ct_label = str(
                summary_row.get("cell_label", "")
            ).strip()

            if ct_id in summary_by_ct:
                summary_by_ct[ct_id]["count"] += count
            else:
                summary_by_ct[ct_id] = {
                    "count": count,
                    "label": ct_label,
                }

        hrapop.setdefault(
            dataset_id,
            {}
        )[method] = summary_by_ct

        annotation_methods.setdefault(
            dataset_id,
            set()
        ).add(method)


# Choose one method per dataset, matching the source workflow preference.
chosen_method = {}

for dataset_id, methods in annotation_methods.items():

    for method in ANNOTATION_METHOD_PREFERENCE:

        if method in methods:
            chosen_method[dataset_id] = method
            break


print(
    f"HRApop dataset records read: {len(hrapop):,}"
)
print(
    f"Datasets with a selected method: {len(chosen_method):,}"
)
print(pd.Series(chosen_method).value_counts())

HRApop dataset records read: 932
Datasets with a selected method: 932
azimuth       692
celltypist    139
popv          101
Name: count, dtype: int64


## 7. Build Anatomogram FTU × dataset counts

In [39]:
# -----------------------------------------------------------------------------
# 7. Build Anatomogram FTU x dataset counts
#
# A cell contributes to an FTU when its AUTHOR CL ID exactly matches one of
# the CT CL IDs defined for that FTU. Anatomogram datasets are included only
# when at least one author-annotated cell matches an FTU-associated CT.
# -----------------------------------------------------------------------------

anatomogram_rows = []


for ftu_purl, ftu in ftu_by_purl.items():

    ftu_label = ftu["ftu_label"]
    organ = ftu["organ"]
    organ_id = ftu["organ_id"]
    ct_by_id = ftu["ct_by_id"]
    ftu_ct_ids = set(ct_by_id)

    if not ftu_ct_ids:
        continue

    # Only Anatomogram datasets from the same organ can contribute.
    organ_author = author.loc[
        author["organ"].eq(organ_id)
    ].copy()

    for dataset_id, dataset_cells in organ_author.groupby(
        "dataset",
        sort=False,
    ):

        matched = dataset_cells.loc[
            dataset_cells["cell_id"].isin(ftu_ct_ids)
        ].copy()

        # A dataset is considered represented for this FTU only when it has
        # at least one author-annotated cell belonging to an FTU-associated CT.
        cell_count = matched["cell"].nunique()

        if cell_count == 0:
            continue

        matched_ct_ids = sorted(
            matched["cell_id"].dropna().unique()
        )

        anatomogram_rows.append({
            "data_source": "Anatomogram",
            "dataset_id": dataset_id,
            "dataset_display_name": dataset_display_name(dataset_id),
            "ftu": ftu_label,
            "ftu_purl": ftu_purl,
            "organ": organ,
            "annotation_method": "author",
            "ftu_ct_count": len(ftu_ct_ids),
            "ftu_ct_ids": "; ".join(sorted(ftu_ct_ids)),
            "ftu_ct_labels": "; ".join(
                ct_by_id[ct_id]
                for ct_id in sorted(ftu_ct_ids)
            ),
            "matched_ct_count": len(matched_ct_ids),
            "matched_ct_ids": "; ".join(matched_ct_ids),
            "matched_ct_labels": "; ".join(
                str(
                    matched.loc[
                        matched["cell_id"].eq(ct_id),
                        "cell_label",
                    ].dropna().iloc[0]
                )
                if not matched.loc[
                    matched["cell_id"].eq(ct_id),
                    "cell_label",
                ].dropna().empty
                else ct_by_id.get(ct_id, "")
                for ct_id in matched_ct_ids
            ),
            "cell_count": float(cell_count),
        })


anatomogram_ftu_counts = pd.DataFrame(
    anatomogram_rows
)


print(
    f"Anatomogram FTU-dataset observations: "
    f"{len(anatomogram_ftu_counts):,}"
)


Anatomogram FTU-dataset observations: 34


## 8. Build HRApop FTU × dataset counts

In [40]:
# -----------------------------------------------------------------------------
# 8. Build HRApop FTU x dataset counts
#
# A dataset contributes once per FTU. The cell count is the sum of the HRApop
# counts for CTs that are members of the FTU's exact CL-ID set.
# -----------------------------------------------------------------------------

hrapop_rows = []


for ftu_purl in matched_ftu_purls:

    ftu = ftu_by_purl[ftu_purl]
    ftu_label = ftu["ftu_label"]
    organ = ftu["organ"]
    ct_by_id = ftu["ct_by_id"]
    ftu_ct_ids = set(ct_by_id)

    for dataset_id in sorted(
        set(ftu_to_datasets[ftu_purl])
    ):

        method = chosen_method.get(
            dataset_id,
            "",
        )

        summary = (
            hrapop.get(dataset_id, {})
            .get(method, {})
        )

        matched_ct_ids = sorted(
            ftu_ct_ids.intersection(summary)
        )

        cell_count = sum(
            summary[ct_id]["count"]
            for ct_id in matched_ct_ids
        )

        hrapop_rows.append({
            "data_source": "HRApop",
            "dataset_id": dataset_id,
            "dataset_display_name": dataset_display_name(dataset_id),
            "ftu": ftu_label,
            "ftu_purl": ftu_purl,
            "organ": organ,
            "annotation_method": method,
            "ftu_ct_count": len(ftu_ct_ids),
            "ftu_ct_ids": "; ".join(sorted(ftu_ct_ids)),
            "ftu_ct_labels": "; ".join(
                ct_by_id[ct_id]
                for ct_id in sorted(ftu_ct_ids)
            ),
            "matched_ct_count": len(matched_ct_ids),
            "matched_ct_ids": "; ".join(matched_ct_ids),
            "matched_ct_labels": "; ".join(
                summary[ct_id]["label"]
                or ct_by_id[ct_id]
                for ct_id in matched_ct_ids
            ),
            "cell_count": float(cell_count),
        })


hrapop_ftu_counts = pd.DataFrame(
    hrapop_rows
)


print(
    f"HRApop FTU-dataset observations: "
    f"{len(hrapop_ftu_counts):,}"
)

HRApop FTU-dataset observations: 2,902


## 9. Combine Anatomogram and HRApop FTU × dataset counts


In [41]:
# -----------------------------------------------------------------------------
# 9. Combine Anatomogram and HRApop FTU x dataset counts
#
# Do not restrict the figure to FTUs present in both resources. The purpose of
# this analysis is to show the distribution of FTU-associated cell counts for
# every FTU represented by either resource. Missing source observations remain
# missing rather than being converted to zero.
# -----------------------------------------------------------------------------

anatomogram_ftus = set(
    anatomogram_ftu_counts.loc[
        anatomogram_ftu_counts["cell_count"] > 0,
        "ftu",
    ]
)

hrapop_ftus = set(
    hrapop_ftu_counts.loc[
        hrapop_ftu_counts["cell_count"] > 0,
        "ftu",
    ]
)

all_ftus = sorted(
    anatomogram_ftus.union(
        hrapop_ftus
    )
)

common_ftus = sorted(
    anatomogram_ftus.intersection(
        hrapop_ftus
    )
)

anatomogram_only_ftus = sorted(
    anatomogram_ftus.difference(
        hrapop_ftus
    )
)

hrapop_only_ftus = sorted(
    hrapop_ftus.difference(
        anatomogram_ftus
    )
)


print(
    f"FTUs represented in either resource: {len(all_ftus)}"
)
print(
    f"FTUs represented in both resources: {len(common_ftus)}"
)
print(
    f"Anatomogram-only FTUs: {len(anatomogram_only_ftus)}"
)
print(
    f"HRApop-only FTUs: {len(hrapop_only_ftus)}"
)


combined = pd.concat(
    [
        anatomogram_ftu_counts,
        hrapop_ftu_counts,
    ],
    ignore_index=True,
)


# Retain only positive cell-count observations for the log-scale distribution.
combined = combined.loc[
    combined["cell_count"] > 0
].copy()


# Add a source-presence label at the FTU level for auditing and hover text.
source_sets = {
    ftu: {
        source
        for source, subset in combined.groupby("data_source")
        if ftu in set(subset["ftu"])
    }
    for ftu in all_ftus
}


def ftu_representation_status(ftu):

    sources = source_sets.get(
        ftu,
        set(),
    )

    if sources == {"Anatomogram", "HRApop"}:
        return "Both"

    if sources == {"Anatomogram"}:
        return "Anatomogram only"

    if sources == {"HRApop"}:
        return "HRApop only"

    return "Unclassified"


combined["ftu_representation_status"] = (
    combined["ftu"].map(
        ftu_representation_status
    )
)


CSV_FILE = (
    OUTPUT_DIR
    / "ftu_dataset_cell_count_distribution_anatomogram_vs_hrapop.csv"
)

combined.to_csv(
    CSV_FILE,
    index=False,
)


print(
    f"Saved combined table:\n{CSV_FILE.resolve()}"
)
print(
    f"Positive-cell observations: {len(combined):,}"
)


FTUs represented in either resource: 13
FTUs represented in both resources: 3
Anatomogram-only FTUs: 3
HRApop-only FTUs: 7
Saved combined table:
/Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/validation/ftu_cell_count_both_sources/ftu_dataset_cell_count_distribution_anatomogram_vs_hrapop.csv
Positive-cell observations: 2,922


## 10. Quality checks

In [42]:
# -----------------------------------------------------------------------------
# 10. Quality checks
# -----------------------------------------------------------------------------

print("Rows by data source:")
print(
    combined["data_source"].value_counts()
)

print("\nDatasets by data source:")
print(
    combined.groupby("data_source")["dataset_id"].nunique()
)

print("\nFTUs by representation status:")
print(
    combined.groupby("ftu_representation_status")["ftu"].nunique()
)

print("\nFTU-level observation counts:")
print(
    combined.groupby(
        ["data_source", "ftu"]
    ).size()
)


Rows by data source:
data_source
HRApop         2888
Anatomogram      34
Name: count, dtype: int64

Datasets by data source:
data_source
Anatomogram     24
HRApop         918
Name: dataset_id, dtype: int64

FTUs by representation status:
ftu_representation_status
Anatomogram only    3
Both                3
HRApop only         7
Name: ftu, dtype: int64

FTU-level observation counts:
data_source  ftu                                      
Anatomogram  alveolus of lung                               4
             islet of Langerhans                            9
             liver lobule                                   6
             nephron                                        5
             renal corpuscle                                5
             thick ascending limb of loop of Henle          5
HRApop       epidermal ridge of digit                      64
             inner medullary collecting duct              535
             liver lobule                                  62
  

## 11. Interactive FTU cell-count distribution

Each FTU is shown as a pair of source-specific distributions. The violin and box summarize the number of cells associated with FTU-linked CTs across datasets. Individual dataset points are retained as a nearly invisible hover layer so the plot stays readable while dataset-level details remain available on hover. FTUs are ordered by organ, and Anatomogram and HRApop are distinguished by color and marker shape.


In [43]:
# -----------------------------------------------------------------------------
# 11. Interactive FTU cell-count distribution
#
# The figure shows the distribution of cells associated with each FTU across
# dataset-level observations.
#
# Only FTUs with actual dataset-level observations in Anatomogram or HRApop
# are included.
#
# Visualization:
#
#   Violin              = distribution across datasets
#   Box                 = interquartile range (IQR)
#   Small points        = individual datasets
#   ● Anatomogram       = median cell count
#   ◆ HRApop            = median cell count
#
# The violin is non-hoverable.
# Only the individual dataset points provide dataset-level hover information.
#
# The distribution is calculated in log10(cell-count) space because the cell
# counts span several orders of magnitude.
# -----------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# Keep only real dataset-level observations
# -----------------------------------------------------------------------------

plot_data = combined.copy()


plot_data["dataset_id"] = (
    plot_data["dataset_id"]
    .astype("string")
    .str.strip()
)


plot_data["cell_count"] = pd.to_numeric(
    plot_data["cell_count"],
    errors="coerce",
)


plot_data = plot_data.loc[
    plot_data["dataset_id"].notna()
    & plot_data["dataset_id"].ne("")
    & plot_data["cell_count"].notna()
    & (plot_data["cell_count"] > 0)
].copy()


# -----------------------------------------------------------------------------
# Restrict to Anatomogram and HRApop
# -----------------------------------------------------------------------------

plot_data = plot_data.loc[
    plot_data["data_source"].isin(
        [
            "Anatomogram",
            "HRApop",
        ]
    )
].copy()


# -----------------------------------------------------------------------------
# Log-transform BEFORE calculating the violin density
# -----------------------------------------------------------------------------

plot_data["log_cell_count"] = np.log10(
    plot_data["cell_count"]
)


# -----------------------------------------------------------------------------
# Keep only FTUs with actual dataset observations
#
# An FTU is included when at least one Anatomogram OR HRApop dataset
# observation exists.
# -----------------------------------------------------------------------------

dataset_supported_ftus = (
    plot_data[
        [
            "organ",
            "ftu",
        ]
    ]
    .drop_duplicates()
)


print(
    "FTUs represented by at least one dataset:",
    len(dataset_supported_ftus),
)


# -----------------------------------------------------------------------------
# Biological order
# -----------------------------------------------------------------------------

organ_priority = {
    "kidney": 0,
    "liver": 1,
    "lung": 2,
    "pancreas": 3,
    "prostate gland": 4,
    "skin": 5,
    "spleen": 6,
}


ftu_order_df = (
    plot_data[
        [
            "organ",
            "ftu",
        ]
    ]
    .drop_duplicates()
)


ftu_order_df["organ_order"] = (
    ftu_order_df["organ"]
    .map(
        lambda value:
            organ_priority.get(
                normalize_organ(value),
                999,
            )
    )
)


ftu_order_df = (
    ftu_order_df
    .sort_values(
        [
            "organ_order",
            "organ",
            "ftu",
        ]
    )
    .reset_index(
        drop=True
    )
)


ftu_order = (
    ftu_order_df[
        "ftu"
    ]
    .tolist()
)


# -----------------------------------------------------------------------------
# Display labels
# -----------------------------------------------------------------------------

ftu_display_labels = {
    row.ftu: (
        f"{str(row.organ).title()} | {row.ftu}"
    )
    for row in ftu_order_df.itertuples()
}


# -----------------------------------------------------------------------------
# Numeric y positions
# -----------------------------------------------------------------------------

position_map = {
    ftu: index * 1.6
    for index, ftu in enumerate(
        ftu_order
    )
}


# Separate the two sources slightly within each FTU row.
source_offset = {
    "Anatomogram": -0.28,
    "HRApop": 0.28,
}


# -----------------------------------------------------------------------------
# Publication-friendly source colors
# -----------------------------------------------------------------------------

source_colors = {
    "Anatomogram": "#D55E00",
    "HRApop": "#0072B2",
}


# Marker shapes used for the visible median markers and dataset points.
source_symbols = {
    "Anatomogram": "circle",
    "HRApop": "diamond",
}


source_display_names = {
    "Anatomogram": "Anatomogram author",
    "HRApop": "HRApop",
}


# -----------------------------------------------------------------------------
# Build the figure
# -----------------------------------------------------------------------------

fig = go.Figure()


for ftu in ftu_order:

    for source in [
        "Anatomogram",
        "HRApop",
    ]:

        subset = plot_data.loc[
            (plot_data["ftu"].eq(ftu))
            & (
                plot_data["data_source"].eq(
                    source
                )
            )
        ].copy()


        if subset.empty:
            continue


        y_value = (
            position_map[ftu]
            + source_offset[source]
        )


        # ---------------------------------------------------------------------
        # Summary statistics
        # ---------------------------------------------------------------------

        dataset_count = (
            subset["dataset_id"]
            .nunique()
        )


        median_cells = (
            subset["cell_count"]
            .median()
        )


        median_log_cells = np.log10(
            median_cells
        )


        # ---------------------------------------------------------------------
        # Violin + box
        #
        # The density is calculated from log10(cell_count).
        #
        # hoverinfo="skip" guarantees that the violin itself is not hoverable.
        # ---------------------------------------------------------------------

        fig.add_trace(
            go.Violin(
                x=subset["log_cell_count"],
                y=np.full(
                    len(subset),
                    y_value,
                ),
                orientation="h",
                name=source_display_names[source],
                legendgroup=source,
                showlegend=False,
                line=dict(
                    color=source_colors[source],
                    width=1.5,
                ),
                fillcolor=source_colors[source],
                opacity=0.25,
                points=False,
                box_visible=True,
                meanline_visible=False,
                width=0.52,
                hoverinfo="skip",
            )
        )


        # ---------------------------------------------------------------------
        # Visible median marker
        #
        #   ● Anatomogram author
        #   ◆ HRApop
        # ---------------------------------------------------------------------

        fig.add_trace(
            go.Scatter(
                x=[
                    median_log_cells
                ],
                y=[
                    y_value
                ],
                mode="markers",
                name=source_display_names[source],
                legendgroup=source,
                showlegend=False,
                marker=dict(
                    color=source_colors[source],
                    symbol=source_symbols[source],
                    size=11,
                    opacity=1.0,
                    line=dict(
                        color="white",
                        width=1.4,
                    ),
                ),
                hovertemplate=(
                    f"<b>{source_display_names[source]} "
                    f"median</b><br>"
                    f"FTU: {ftu_display_labels[ftu]}<br>"
                    f"Median cells: {median_cells:,.0f}<br>"
                    f"Datasets: {dataset_count:,}"
                    "<extra></extra>"
                ),
            )
        )


        # ---------------------------------------------------------------------
        # Visible dataset-level observations
        #
        # Each point represents one dataset.
        #
        # The points are deliberately small and semi-transparent so they show
        # the underlying observations without overwhelming the distribution.
        # ---------------------------------------------------------------------

        if len(subset) > 1:

            jitter = np.linspace(
                -0.13,
                0.13,
                num=len(subset),
            )

        else:

            jitter = np.array(
                [0.0]
            )


        customdata = subset[
            [
                "dataset_id",
                "dataset_display_name",
                "data_source",
                "organ",
                "ftu",
                "annotation_method",
                "ftu_ct_count",
                "ftu_ct_labels",
                "matched_ct_count",
                "matched_ct_labels",
                "ftu_representation_status",
                "cell_count",
            ]
        ].fillna("").to_numpy()


        fig.add_trace(
            go.Scattergl(
                x=subset["log_cell_count"],
                y=(
                    np.full(
                        len(subset),
                        y_value,
                    )
                    + jitter
                ),
                mode="markers",
                name=source_display_names[source],
                legendgroup=source,
                showlegend=False,
                marker=dict(
                    color=source_colors[source],
                    symbol=source_symbols[source],
                    size=6,
                    opacity=0.20,
                    line=dict(
                        color=source_colors[source],
                        width=0.3,
                    ),
                ),
                customdata=customdata,
                hovertemplate=(
                    "<b>Dataset</b><br>"
                    "Dataset: %{customdata[1]}<br>"
                    "Dataset ID: %{customdata[0]}<br>"
                    "Source: %{customdata[2]}<br>"
                    "Organ: %{customdata[3]}<br>"
                    "FTU: %{customdata[4]}<br>"
                    "Annotation method: %{customdata[5]}<br>"
                    "FTU CT count: %{customdata[6]}<br>"
                    "Associated CTs: %{customdata[7]}<br>"
                    "Observed CT count: %{customdata[8]}<br>"
                    "Observed CTs: %{customdata[9]}<br>"
                    "FTU representation: %{customdata[10]}<br>"
                    "Cells: %{customdata[11]:,.0f}"
                    "<extra></extra>"
                ),
            )
        )


# -----------------------------------------------------------------------------
# Explicit source legend
#
# These marker-only traces ensure that the legend uses exactly the same
# symbols as the visible median markers:
#
#   ● Anatomogram author
#   ◆ HRApop
# -----------------------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        name="Anatomogram author",
        legendgroup="Anatomogram",
        showlegend=True,
        marker=dict(
            color=source_colors["Anatomogram"],
            symbol="circle",
            size=9,
        ),
    )
)


fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        name="HRApop",
        legendgroup="HRApop",
        showlegend=True,
        marker=dict(
            color=source_colors["HRApop"],
            symbol="diamond",
            size=9,
        ),
    )
)


# -----------------------------------------------------------------------------
# Y-axis
# -----------------------------------------------------------------------------

fig.update_yaxes(
    tickmode="array",
    tickvals=[
        position_map[ftu]
        for ftu in ftu_order
    ],
    ticktext=[
        ftu_display_labels[ftu]
        for ftu in ftu_order
    ],
    title="Organ | functional tissue unit",
    showgrid=False,
    zeroline=False,
    tickfont=dict(
        size=10
    ),
)


# -----------------------------------------------------------------------------
# Organ separators
# -----------------------------------------------------------------------------

for index in range(
    len(ftu_order_df) - 1
):

    current_organ = normalize_organ(
        ftu_order_df.iloc[
            index
        ]["organ"]
    )


    next_organ = normalize_organ(
        ftu_order_df.iloc[
            index + 1
        ]["organ"]
    )


    if current_organ != next_organ:

        current_ftu = (
            ftu_order_df.iloc[
                index
            ]["ftu"]
        )


        next_ftu = (
            ftu_order_df.iloc[
                index + 1
            ]["ftu"]
        )


        separator_y = (
            position_map[current_ftu]
            + position_map[next_ftu]
        ) / 2


        fig.add_shape(
            type="line",
            xref="paper",
            x0=0,
            x1=1,
            yref="y",
            y0=separator_y,
            y1=separator_y,
            line=dict(
                color="#D9D9D9",
                width=1,
            ),
        )


# -----------------------------------------------------------------------------
# X-axis
#
# The plotted coordinate is log10(cell_count), but the tick labels show
# the original cell-count scale.
# -----------------------------------------------------------------------------

min_log = (
    plot_data["log_cell_count"]
    .min()
)


max_log = (
    plot_data["log_cell_count"]
    .max()
)


min_exponent = int(
    np.floor(
        min_log
    )
)


max_exponent = int(
    np.ceil(
        max_log
    )
)


tick_exponents = np.arange(
    min_exponent,
    max_exponent + 1,
)


tick_labels = []


for exponent in tick_exponents:

    value = 10 ** exponent

    if value >= 1_000_000:

        tick_labels.append(
            f"{value / 1_000_000:g}M"
        )

    elif value >= 1_000:

        tick_labels.append(
            f"{value / 1_000:g}k"
        )

    else:

        tick_labels.append(
            f"{int(value)}"
        )


fig.update_xaxes(
    tickmode="array",
    tickvals=tick_exponents,
    ticktext=tick_labels,
    title=(
        "#cells assigned to CTs per FTU per dataset"
    ),
    showgrid=True,
    gridcolor="#E6E6E6",
    zeroline=False,
)


# -----------------------------------------------------------------------------
# Final layout
# -----------------------------------------------------------------------------

fig.update_layout(
    title=(
        "Distribution of cells per CTs per FTU across "
        "Anatomogram and HRApop datasets"
    ),
    template="simple_white",
    width=1450,
    height=max(
        850,
        75 * len(ftu_order),
    ),
    margin=dict(
        l=390,
        r=40,
        t=90,
        b=100,
    ),
    hovermode="closest",
    hoverlabel=dict(
        bgcolor="white",
        font_size=12,
    ),
    legend=dict(
        title="Data source",
        orientation="h",
        yanchor="bottom",
        y=1.01,
        xanchor="right",
        x=1,
    ),
)


# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

HTML_FILE = (
    OUTPUT_DIR
    / "ftu_cell_count_distribution_anatomogram_vs_hrapop.html"
)


fig.write_html(
    HTML_FILE,
    include_plotlyjs="cdn",
)


print(
    f"Saved interactive figure:\n"
    f"{HTML_FILE.resolve()}"
)


print(
    "\nFTUs plotted:",
    len(ftu_order),
)


print(
    "\nDataset-level observations by source:"
)


print(
    plot_data[
        "data_source"
    ]
    .value_counts()
    .to_string()
)


fig.show()

FTUs represented by at least one dataset: 13


Saved interactive figure:
/Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/validation/ftu_cell_count_both_sources/ftu_cell_count_distribution_anatomogram_vs_hrapop.html

FTUs plotted: 13

Dataset-level observations by source:
data_source
HRApop         2888
Anatomogram      34


## 12. Export a small FTU summary table for manuscript reporting

In [44]:
# -----------------------------------------------------------------------------
# 12. Export an FTU summary table for manuscript reporting
# -----------------------------------------------------------------------------

summary = (
    combined
    .groupby(
        [
            "data_source",
            "organ",
            "ftu",
            "ftu_representation_status",
        ],
        as_index=False,
    )
    .agg(
        datasets=("dataset_id", "nunique"),
        median_cells=("cell_count", "median"),
        min_cells=("cell_count", "min"),
        max_cells=("cell_count", "max"),
    )
)


summary_file = (
    OUTPUT_DIR
    / "ftu_cell_count_distribution_summary.csv"
)

summary.to_csv(
    summary_file,
    index=False,
)


display(summary)

print(
    f"Saved summary:\n{summary_file.resolve()}"
)


,data_source,organ,ftu,ftu_representation_status,datasets,median_cells,min_cells,max_cells
0,Anatomogram,Liver,liver lobule,Both,6,640.5,306.0,1020.0
1,Anatomogram,Pancreas,islet of Langerhans,Anatomogram only,9,168.0,65.0,230.0
2,Anatomogram,kidney,nephron,Both,5,2046.0,1092.0,2440.0
3,Anatomogram,kidney,renal corpuscle,Both,5,1292.0,820.0,1944.0
4,Anatomogram,kidney,thick ascending limb of loop of Henle,Anatomogram only,5,626.0,393.0,1994.0
5,Anatomogram,lung,alveolus of lung,Anatomogram only,4,134.5,11.0,315.0
6,HRApop,Liver,liver lobule,Both,62,283.5,2.0,17949.0
7,HRApop,Pancreas,pancreatic acinus,HRApop only,13,120.0,6.0,5874.0
8,HRApop,kidney,inner medullary collecting duct,HRApop only,535,222.0,1.0,48742.0
9,HRApop,kidney,loop of Henle ascending limb thin segment,HRApop only,371,265.0,1.0,39010.0


Saved summary:
/Users/sbidanta/projects/HRA/Publication/hra-ftu-explorer-supporting-information/validation/ftu_cell_count_both_sources/ftu_cell_count_distribution_summary.csv
